In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Create separate folders for raw and cleaned data
BASE_DIR = Path("/content/Task2_Data_Cleaning")
RAW_DIR = BASE_DIR / "raw"
OUTPUT_DIR = BASE_DIR / "output"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Download the original UCI Online Retail dataset
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

import urllib.request
import zipfile

zip_path = RAW_DIR / "online_retail.zip"
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

# Locate and load the original Excel file
excel_files = list(RAW_DIR.glob("*.xlsx"))
print("Excel files found:", [f.name for f in excel_files])

df = pd.read_excel(excel_files[0])

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
display(df.head())


Excel files found: ['Online Retail.xlsx']
Dataset loaded successfully!
Rows and columns: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:

# Step 3: Initial Data Quality Audit

print("Dataset shape:", df.shape)

print("\n--- Column Names ---")
print(df.columns.tolist())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

print("\n--- Duplicate Rows ---")
print("Total duplicate rows:", df.duplicated().sum())

print("\n--- Statistical Summary ---")
display(df.describe(include="all").T)

print("\n--- Sample Records ---")
display(df.head(10))


Dataset shape: (541909, 8)

--- Column Names ---
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

--- Data Types ---
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

--- Missing Values ---
Description      1454
CustomerID     135080
dtype: int64

--- Duplicate Rows ---
Total duplicate rows: 5268

--- Statistical Summary ---


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Sample Records ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [3]:

# Step 4: Save Before-Cleaning Metrics

before_metrics = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Cells",
        "Duplicate Rows",
        "Invalid Quantity (<= 0)",
        "Invalid UnitPrice (<= 0)"
    ],
    "Value": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int((df["Quantity"] <= 0).sum()),
        int((df["UnitPrice"] <= 0).sum())
    ]
})

display(before_metrics)

before_metrics.to_csv(
    OUTPUT_DIR / "before_cleaning_metrics.csv",
    index=False
)

print("Before-cleaning metrics saved!")


,Metric,Value
0,Total Rows,541909
1,Total Columns,8
2,Missing Cells,136534
3,Duplicate Rows,5268
4,Invalid Quantity (<= 0),10624
5,Invalid UnitPrice (<= 0),2517


Before-cleaning metrics saved!


In [4]:

# Step 5: Remove Exact Duplicate Rows

rows_before = len(df)
duplicate_count = int(df.duplicated().sum())

df_clean = df.drop_duplicates().copy()

rows_after = len(df_clean)
removed_count = rows_before - rows_after

print("Rows before cleaning:", rows_before)
print("Duplicate rows found:", duplicate_count)
print("Duplicate rows removed:", removed_count)
print("Rows after cleaning:", rows_after)

assert not df_clean.duplicated().any(), "Duplicates still exist!"

print("\nExact duplicate rows removed successfully.")


Rows before cleaning: 541909
Duplicate rows found: 5268
Duplicate rows removed: 5268
Rows after cleaning: 536641

Exact duplicate rows removed successfully.


In [5]:

# Step 6: Analyse Missing Values

missing_report = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (
        df_clean.isnull().sum() / len(df_clean) * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

display(missing_report)

missing_report.to_csv(
    OUTPUT_DIR / "missing_values_report.csv"
)

print("Missing-value report saved!")


,Missing_Count,Missing_Percentage
CustomerID,135037,25.16
Description,1454,0.27


Missing-value report saved!


In [6]:

# Step 7: Standardize Data Types

# Convert date column to datetime format
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"], errors="coerce"
)

# Convert numeric columns safely
df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"], errors="coerce"
)

df_clean["UnitPrice"] = pd.to_numeric(
    df_clean["UnitPrice"], errors="coerce"
)

# Remove leading/trailing spaces from text columns
for col in ["Description", "Country", "StockCode"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("string").str.strip()

print("Updated data types:")
print(df_clean.dtypes)

print("\nInvalid/missing values after conversion:")
print(df_clean[["InvoiceDate", "Quantity", "UnitPrice"]].isnull().sum())


Updated data types:
InvoiceNo              object
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country        string[python]
dtype: object

Invalid/missing values after conversion:
InvoiceDate    0
Quantity       0
UnitPrice      0
dtype: int64


In [7]:

# Step 8: Handle Invalid Values

rows_before = len(df_clean)

# Count invalid values before removal
invalid_date = df_clean["InvoiceDate"].isna().sum()
invalid_quantity = df_clean["Quantity"].isna().sum()
invalid_price = df_clean["UnitPrice"].isna().sum()

nonpositive_quantity = (df_clean["Quantity"] <= 0).sum()
nonpositive_price = (df_clean["UnitPrice"] <= 0).sum()

print("Missing/invalid invoice dates:", invalid_date)
print("Missing/invalid quantities:", invalid_quantity)
print("Missing/invalid prices:", invalid_price)
print("Zero/negative quantities:", nonpositive_quantity)
print("Zero/negative prices:", nonpositive_price)

# Keep rows with valid dates and positive quantity and price
df_clean = df_clean.dropna(
    subset=["InvoiceDate", "Quantity", "UnitPrice"]
).copy()

df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

print("\nRows before:", rows_before)
print("Rows after:", len(df_clean))
print("Rows removed:", rows_before - len(df_clean))


Missing/invalid invoice dates: 0
Missing/invalid quantities: 0
Missing/invalid prices: 0
Zero/negative quantities: 10587
Zero/negative prices: 2512

Rows before: 536641
Rows after: 524878
Rows removed: 11763


In [8]:

# Step 9: Detect Outliers using IQR

numeric_cols = ["Quantity", "UnitPrice"]
outlier_results = []

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    )

    outlier_results.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": int(outlier_mask.sum()),
        "Outlier_Percentage": round(
            outlier_mask.mean() * 100, 2
        )
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report)

outlier_report.to_csv(
    OUTPUT_DIR / "iqr_outlier_report.csv",
    index=False
)

print("IQR outlier report saved!")


,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,Quantity,1.00,11.00,10.00,-14.00,26.00,27111,5.17
1,UnitPrice,1.25,4.13,2.88,-3.07,8.45,37827,7.21


IQR outlier report saved!


In [9]:

# Step 10: Before-and-After Data Quality Comparison

after_metrics = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Cells",
        "Duplicate Rows",
        "Invalid Quantity (<= 0)",
        "Invalid UnitPrice (<= 0)"
    ],
    "Value": [
        len(df_clean),
        len(df_clean.columns),
        int(df_clean.isnull().sum().sum()),
        int(df_clean.duplicated().sum()),
        int((df_clean["Quantity"] <= 0).sum()),
        int((df_clean["UnitPrice"] <= 0).sum())
    ]
})

before_table = before_metrics.rename(
    columns={"Value": "Before"}
)
after_table = after_metrics.rename(
    columns={"Value": "After"}
)

comparison = before_table.merge(after_table, on="Metric")
comparison["Change"] = comparison["After"] - comparison["Before"]

display(comparison)

comparison.to_csv(
    OUTPUT_DIR / "before_after_comparison.csv",
    index=False
)

print("Before-and-after comparison saved!")


,Metric,Before,After,Change
0,Total Rows,541909,524878,-17031
1,Total Columns,8,8,0
2,Missing Cells,136534,132186,-4348
3,Duplicate Rows,5268,0,-5268
4,Invalid Quantity (<= 0),10624,0,-10624
5,Invalid UnitPrice (<= 0),2517,0,-2517


Before-and-after comparison saved!


In [10]:

# Step 11: Save Cleaned Dataset

cleaned_csv_path = OUTPUT_DIR / "online_retail_cleaned.csv"

df_clean.to_csv(cleaned_csv_path, index=False)

print("Cleaned dataset saved successfully!")
print("Saved location:", cleaned_csv_path)
print("Final rows:", len(df_clean))
print("Final columns:", len(df_clean.columns))

# Verify the saved file
df_check = pd.read_csv(cleaned_csv_path)

print("\nSaved file verification:")
print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))
print("Duplicate rows:", df_check.duplicated().sum())


Cleaned dataset saved successfully!
Saved location: /content/Task2_Data_Cleaning/output/online_retail_cleaned.csv
Final rows: 524878
Final columns: 8


/tmp/ipykernel_671/802702665.py:13: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(cleaned_csv_path)



Saved file verification:
Rows: 524878
Columns: 8
Duplicate rows: 0


In [11]:

# Step 12: Generate Data Quality Audit Report

report = f"""# Task 2: Data Quality Audit Report

## 1. Project Overview
This project audits and cleans the UCI Online Retail dataset.
The original raw dataset is preserved separately.

## 2. Dataset Summary
- Original rows: {len(df)}
- Cleaned rows: {len(df_clean)}
- Original columns: {len(df.columns)}
- Cleaned columns: {len(df_clean.columns)}
- Rows removed: {len(df) - len(df_clean)}

## 3. Cleaning Steps
1. Inspected missing values, data types, and duplicate rows.
2. Removed exact duplicate records.
3. Standardized invoice dates and numeric columns.
4. Trimmed extra spaces from selected text columns.
5. Removed rows with missing or invalid invoice dates,
   quantities, or unit prices.
6. Detected Quantity and UnitPrice outliers using the IQR method.
7. Saved the cleaned dataset separately from the raw file.

## 4. Cleaning Rationale
- Exact duplicates were removed to prevent double counting.
- Invalid dates were removed because they cannot support reliable
  date-based analysis.
- Non-positive quantities and prices were excluded from this
  positive-sales dataset; returns and cancellations require
  separate business treatment.
- IQR outliers were reported, not automatically removed, because
  extreme values may represent genuine transactions.
- Missing values were inspected; remaining missing values should
  be reviewed before downstream analysis.

## 5. Outlier Analysis
The IQR outlier counts are available in `iqr_outlier_report.csv`.
Outliers were retained unless they were invalid under the documented
rules above.

## 6. Deliverables
- `online_retail_cleaned.csv`
- `before_cleaning_metrics.csv`
- `before_after_comparison.csv`
- `missing_values_report.csv`
- `iqr_outlier_report.csv`

## 7. Limitations
The cleaning rules focus on positive sales transactions.
The original raw dataset remains unchanged. All cleaning steps can
be rerun using the notebook cells in order.

## 8. Conclusion
The workflow documents data quality issues, cleaning decisions,
outlier detection, and before-and-after metrics for reproducible
data preparation.
"""

report_path = OUTPUT_DIR / "Data_Quality_Audit_Report.md"
report_path.write_text(report, encoding="utf-8")

print("Audit report created:", report_path)
print(report)


Audit report created: /content/Task2_Data_Cleaning/output/Data_Quality_Audit_Report.md
# Task 2: Data Quality Audit Report

## 1. Project Overview
This project audits and cleans the UCI Online Retail dataset.
The original raw dataset is preserved separately.

## 2. Dataset Summary
- Original rows: 541909
- Cleaned rows: 524878
- Original columns: 8
- Cleaned columns: 8
- Rows removed: 17031

## 3. Cleaning Steps
1. Inspected missing values, data types, and duplicate rows.
2. Removed exact duplicate records.
3. Standardized invoice dates and numeric columns.
4. Trimmed extra spaces from selected text columns.
5. Removed rows with missing or invalid invoice dates,
   quantities, or unit prices.
6. Detected Quantity and UnitPrice outliers using the IQR method.
7. Saved the cleaned dataset separately from the raw file.

## 4. Cleaning Rationale
- Exact duplicates were removed to prevent double counting.
- Invalid dates were removed because they cannot support reliable
  date-based analysis

In [12]:

# Step 13: Package Task 2 Deliverables

import shutil
from google.colab import files

# Create a ZIP containing all output reports and cleaned data
zip_path = shutil.make_archive(
    "/content/Task2_Data_Cleaning_Deliverables",
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("ZIP created successfully:", zip_path)
print("Files included:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print("-", file_path.name)

# Download ZIP to your computer
files.download(zip_path)


ZIP created successfully: /content/Task2_Data_Cleaning_Deliverables.zip
Files included:
- Data_Quality_Audit_Report.md
- before_after_comparison.csv
- before_cleaning_metrics.csv
- iqr_outlier_report.csv
- missing_values_report.csv
- online_retail_cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>